# Tests excluded from CICD Automation

⚠️ As much as possible test should be made for CICD even if a manual version is also made that is less pass fail and more an is this a potential area to check for identifying an issue.

@pytest.mark.manual
can be used as our mark to specify tests only to run manually.

This could be useful for exploration, and identifying problems.
It may be tests with much stricter data expectations

There is also a displosable scratch runner for one off tests not for source control.



## Maybe useful marker string for top left
"not dev_skip and not freshness"

Env Setup

In [0]:
# MAGIC %pip install pytest
# because doesnt get pytest from toml???
import pytest
import sys
import os

# We need the catalog for our temp storage volume
# we also need it if data tests
# Get catalog from job
dbutils.widgets.text("job.catalog", "dev_catalog") # should default to bundle value if exists
catalog = dbutils.widgets.get("job.catalog")
print(catalog)

# for db access without deploy
dbutils.widgets.text("job.schema_prefix", "") # e.g local user environment
schema_prefix = dbutils.widgets.get("job.schema_prefix") #remember username trailing underscore
print(schema_prefix)

#  we cant get bundle vars from spark unless the bundle is deployed so we need logic so we need a fall back for running locally in personal area

#  these var passings shouldnt be the best way need to see a working example
# So pytest can get it from the environment
os.environ["CATALOG"] = catalog
os.environ["SCHEMA_PREFIX"] = schema_prefix
# This looks 'up' one level from this notebook to find your project root
# using toml now
# repo_root = os.path.abspath('..') # sys.path.append(f"{repo_root}/src")

# Prevent __pycache__ creation
sys.dont_write_bytecode = True

# print(f"Project root set to: {repo_root}")
print("Setup run")

Run Tests

In [0]:
# ===== WIDGET SETUP =====
#dbutils.widgets.removeAll()  # Uncomment to reset

dbutils.widgets.multiselect(
    "test_suites",
    "data-quality-tests",
    ["unit-tests", "data-quality-tests", "integration-tests"],
    "Test Suites"
)

dbutils.widgets.text(
    "specific_test",
    "test_module.py::TestClass::test_method",
    "Specific Test"
)

dbutils.widgets.text(
    "markers",
    "manual",
    "Markers"
)

# ===== RUN TESTS =====
import pytest

# Get widget values
test_suites = dbutils.widgets.get("test_suites").split(",")
specific_test = dbutils.widgets.get("specific_test").strip()
markers = dbutils.widgets.get("markers").strip()

# Build pytest arguments
pytest_args = []

# Add test suites
pytest_args.extend(test_suites)

# Add specific test if provided
if specific_test:
    pytest_args.append(specific_test)

# Add markers if provided
if markers:
    pytest_args.extend(["-m", markers])

# Add standard options
pytest_args.extend(["-v", "-s", "--tb=short"])

# Run
print(f"Running: pytest {' '.join(pytest_args)}\n")
exit_code = pytest.main(pytest_args)

print(f"\n{'✅ PASSED' if exit_code == 0 else '❌ FAILED'} (exit code: {exit_code})")